In [1]:
import os
from dotenv import load_dotenv
import json


from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


from functions.elasticsearch_utils.return_vectordb_full_text_and_questions import return_vectordb_full_text_and_questions


from autogen_core import AgentId, MessageContext, RoutedAgent, SingleThreadedAgentRuntime, message_handler
from functions.autogen_utils.RouterAgent import RouterAgent
from functions.autogen_utils.DeepSeekAgent import DeepSeekAgent
from functions.autogen_utils.TimeSeriesAgent import TimeSeriesAgent
from functions.autogen_utils.OuterAgent import OuterAgent
from functions.autogen_utils.Message import Message

In [2]:
load_dotenv()
# Check if variables are correctly loaded from .env
AZURE_OPENAI_API_KEY_2 = os.getenv('AZURE_OPENAI_API_KEY')
if not AZURE_OPENAI_API_KEY_2:
    raise ValueError("AZURE_OPENAI_API_KEY not found in environment variables")

DEPLOYMENT_NAME_LLM = os.getenv('DEPLOYMENT_NAME_LLM')
if not DEPLOYMENT_NAME_LLM:
    raise ValueError("DEPLOYMENT_NAME_LLM not found in environment variables")

API_VERSION = os.getenv('API_VERSION')
if not API_VERSION:
    raise ValueError("API_VERSION not found in environment variables")
    
AZURE_ENDPOINT_LLM = os.getenv('AZURE_ENDPOINT_LLM')
if not AZURE_ENDPOINT_LLM:
    raise ValueError("AZURE_ENDPOINT_LLM not found in environment variables")

EMBEDDING_KEY = os.getenv('EMBEDDING_KEY')
if not EMBEDDING_KEY:
    raise ValueError("EMBEDDING_KEY not found in environment variables")

DEPLOYMENT_NAME_EMBEDDING = os.getenv('DEPLOYMENT_NAME_EMBEDDING')
if not DEPLOYMENT_NAME_EMBEDDING:
    raise ValueError("DEPLOYMENT_NAME_EMBEDDING not found in environment variables")
    
AZURE_ENDPOINT_EMBEDDING = os.getenv('AZURE_ENDPOINT_EMBEDDING')
if not AZURE_ENDPOINT_EMBEDDING:
    raise ValueError("AZURE_ENDPOINT_EMBEDDING not found in environment variables")

API_BASE_EMBEDDING = os.getenv('API_BASE_EMBEDDING')
if not API_BASE_EMBEDDING:
    raise ValueError("API_BASE_EMBEDDING not found in environment variables")

ELASTICSEARCH_USER = os.getenv('ELASTICSEARCH_USER')
if not ELASTICSEARCH_USER:
    raise ValueError("ELASTICSEARCH_USER not found in environment variables")

ELASTICSEARCH_PASSWORD = os.getenv('ELASTICSEARCH_PASSWORD')
if not ELASTICSEARCH_PASSWORD:
    raise ValueError("ELASTICSEARCH_PASSWORD not found in environment variables")

ELASTICSEARCH_API_KEY = os.getenv('ELASTICSEARCH_API_KEY')
if not ELASTICSEARCH_API_KEY:
    raise ValueError("ELASTICSEARCH_API_KEY not found in environment variables")

ELASTICSEARCH_ENDPOINT = os.getenv('ELASTICSEARCH_ENDPOINT')
if not ELASTICSEARCH_ENDPOINT:
    raise ValueError("ELASTICSEARCH_ENDPOINT not found in environment variables")

In [3]:
# Get information about how the LLM should answer the questions
target_audience = 'PhD student'
answer_tone = 'Professional and Clear'
answer_length = '4 paragraphs'

# Get necessary information about the collection in the vector db
index_name_collection_full_text = 'collection_full_text'
index_name_collection_questions_text = 'collection_questions_text'

# Connect with elastic seach and langchain
dict_vectordb = return_vectordb_full_text_and_questions(
    embedding_key = EMBEDDING_KEY,
    deployment_name_embedding = DEPLOYMENT_NAME_EMBEDDING,
    azure_endpoint_embedding = AZURE_ENDPOINT_EMBEDDING,
    elasticsearch_endpoint = ELASTICSEARCH_ENDPOINT,
    elasticsearch_user = ELASTICSEARCH_USER,
    elasticsearch_password = ELASTICSEARCH_PASSWORD,
)

In [4]:
# Define LLM model
llm = AzureChatOpenAI(
    temperature = 0,
    model_name = "gpt-4o-mini",
    deployment_name = DEPLOYMENT_NAME_LLM,  
    api_version = API_VERSION,
    azure_endpoint = AZURE_ENDPOINT_LLM
)

### Statefully manage chat history ###
store = {} # This dictionary will retain all chat history

In [5]:
# Define prompts

################################ RouterAgent ###################################################
system_template_RouterAgent = '''You are a helpful virtual assistant router that will decide which virtal assistant can help anserwing the user question.
You will answer only the number of the description that is most related to the user question.
You should answer just a number.

Descriptions:
1. A Survey of Time Series Foundation Models: Generalizing Time Series Representation with Large Language Model
Time series data are ubiquitous across various domains, making time series analysis critically important. Traditional time series models
are task-specific, featuring singular functionality and limited generalization capacity. Recently, large language foundation models have
unveiled their remarkable capabilities for cross-task transferability, zero-shot/few-shot learning, and decision-making explainability.
This success has sparked interest in the exploration of foundation models to solve multiple time series challenges simultaneously. There
are two main research lines, namely pre-training foundation models from scratch for time series and adapting large language
foundation models for time series. They both contribute to the development of a unified model that is highly generalizable, versatile,
and comprehensible for time series analysis. This survey offers a 3E analytical framework for comprehensive examination of related
research. Specifically, we examine existing works from three dimensions, namely Effectiveness, Efficiency and Explainability. In
each dimension, we focus on discussing how related works devise tailored solution by considering unique challenges in the realm of
time series.Furthermore, we provide a domain taxonomy to help followers keep up with the domain-specific advancements. In addition,
we introduce extensive resources to facilitate the field’s development, including datasets, open-source, time series libraries. A GitHub
repository is also maintained for resource updates (https://github.com/start2020/Awesome-TimeSeries-LLM-FM).


2. DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning
We introduce our first-generation reasoning models, DeepSeek-R1-Zero and DeepSeek-R1.
DeepSeek-R1-Zero, a model trained via large-scale reinforcement learning (RL) without supervised
fine-tuning (SFT) as a preliminary step, demonstrates remarkable reasoning capabilities.
Through RL, DeepSeek-R1-Zero naturally emerges with numerous powerful and intriguing
reasoning behaviors. However, it encounters challenges such as poor readability, and language
mixing. To address these issues and further enhance reasoning performance, we introduce
DeepSeek-R1, which incorporates multi-stage training and cold-start data before RL. DeepSeek-
R1 achieves performance comparable to OpenAI-o1-1217 on reasoning tasks. To support the
research community, we open-source DeepSeek-R1-Zero, DeepSeek-R1, and six dense models
(1.5B, 7B, 8B, 14B, 32B, 70B) distilled from DeepSeek-R1 based on Qwen and Llama.


user question:
{user_question}
'''

################################ OuterAgent ###################################################
system_template_OuterAgent = """ 
You are a helpful virtual assistant specialized in answering academic questions.
Your answers must be based on the context that you will receive.
You will answer considering the parameters:
- Target audience: {target_audience}
- Tone: {answer_tone}
- Length: {answer_length}

context:
{text_context}


User question:
{user_question}
"""

In [6]:
# initializes a runtime environment for multiple agents, registers them, and starts the runtime.
runtime = SingleThreadedAgentRuntime()
await RouterAgent.register(runtime, "router_agent", lambda: RouterAgent(
    description = "RouterAgent",
    system_template = system_template_RouterAgent,
    llm = llm,
))
await TimeSeriesAgent.register(runtime, "TimeSeries_agent", lambda: TimeSeriesAgent(
    description = "TimeSeriesAgent",
    dict_vectordb = dict_vectordb,
))
await DeepSeekAgent.register(runtime, "DeepSeek_agent", lambda: DeepSeekAgent(
    description = "DeepSeekAgent",
    dict_vectordb = dict_vectordb,
))
await OuterAgent.register(
    runtime,
    "outer_agent",
    lambda: OuterAgent(
        description = "OuterAgent",
        llm = llm,
        session_id = 'Teste_01',
        store = store,
        system_template = system_template_OuterAgent,
        router_agent_id = "router_agent",
        TimeSeries_agent_id = "TimeSeries_agent",
        DeepSeek_agent_id = "DeepSeek_agent",
        target_audience = target_audience,
        answer_tone = answer_tone,
        answer_length = answer_length,
    ),
)

# The runtime.start() command launches the runtime, enabling the agents to function and interact as intended.
runtime.start()

TESTING: Inside OuterAgent.
TESTING: Inside RouterAgent.
TESTING: Router: 1.
TESTING: rout_to_agent: 1.
-----------------------------------------------------------------------

TESTING: Inside TimeSeriesAgent.
TESTING: TimeSeriesAgent: {'scores_collection_full_text': {1: {'score': 0.8316679, 'page_content': 'pre-training stage. The catastrophic forgetting may occur, where LLM loses its previous capacities. Another drawback is\nthe requirement of large training datasets and heavy computational resources. For downstream tasks with small datasets,\nfully fine-tuning LLM may have unstable performance or over-fit.\nTo alleviate these defects, some works leverage efficient tuning, aiming to reduce the number of trainable parameters\nwhile retaining a good performance as possible. FPT [ 215], TEMPO [ 25] and LLM4TS [ 27] all freeze the majority of the\nLLM parameters and only update the minority ones in general time series forecasting. This method can retain the major\nknowledge and capacitie

In [7]:
# Send and receive a message
# Give an id for the agent that will communicate with the user
outer_agent_id = AgentId("outer_agent", "default")

# Send message to chat
user_message = "How can I use LLM models to help in time series predictions?"
response = await runtime.send_message(Message(content=user_message), outer_agent_id)

print(f"Final response: {response.content}")

Final response: Utilizing Large Language Models (LLMs) for time series predictions involves adapting their capabilities to effectively process and analyze temporal data. The first step is to choose an appropriate adaptation paradigm, which can be broadly categorized into embedding-visible and text-visible LLM adaptations. The embedding-visible approach involves redesigning the LLM to directly perceive time series embeddings, while the text-visible approach reformulates time series tasks into a textual format that the LLM can understand. This transformation allows numerical time series data to be represented as strings, enabling seamless integration into prompts that activate the LLM's predictive capabilities.

Once the adaptation paradigm is selected, the next step is to prepare the time series data for input into the LLM. This involves tokenizing the time series data and aligning it with the semantic space of the LLM. Techniques such as temporal pattern recognition and multi-modal dat

In [8]:
# Send and receive a message
# Give an id for the agent that will communicate with the user
outer_agent_id = AgentId("outer_agent", "default")

# Send message to chat
user_message = "Could you explain better the first step?"
response = await runtime.send_message(Message(content=user_message), outer_agent_id)

print(f"Final response: {response.content}")

Final response: The first step in utilizing Large Language Models (LLMs) for time series predictions involves selecting an appropriate adaptation paradigm that allows the model to effectively process and analyze temporal data. This step is crucial because LLMs are primarily designed for natural language processing, and time series data, which consists of sequential numerical values, requires a different approach to be effectively interpreted by these models.

There are two main adaptation paradigms to consider: embedding-visible and text-visible adaptations. The embedding-visible approach focuses on transforming the time series data into a format that can be directly fed into the LLM as embeddings. This involves creating vector representations of the time series data that capture its temporal characteristics. Techniques such as Fourier transforms, wavelet transforms, or recurrent neural networks can be employed to generate these embeddings, which can then be input into the LLM for furt

In [9]:
# Send and receive a message
# Give an id for the agent that will communicate with the user
outer_agent_id = AgentId("outer_agent", "default")

# Send message to chat
user_message = "Could you explian me DeepSeek-R1-Zero?"
response = await runtime.send_message(Message(content=user_message), outer_agent_id)

print(f"Final response: {response.content}")

Final response: DeepSeek-R1-Zero is a first-generation reasoning model developed by DeepSeek-AI, designed to enhance reasoning capabilities in large language models (LLMs) through a novel approach based on large-scale reinforcement learning (RL). Unlike traditional models that often rely on supervised fine-tuning (SFT) as a preliminary step, DeepSeek-R1-Zero is trained directly using RL, which allows it to autonomously develop reasoning behaviors without the constraints of supervised data. This unique training methodology enables the model to exhibit remarkable reasoning capabilities, emerging with a variety of powerful and intriguing reasoning behaviors.

The training process of DeepSeek-R1-Zero involves initiating RL directly from a base model, specifically DeepSeek-V3-Base, and employing a reinforcement learning framework known as GRPO (Shao et al., 2024). Throughout this process, the model undergoes thousands of RL steps, during which it learns to solve increasingly complex reasoni

In [11]:
store

{'Teste_01': InMemoryChatMessageHistory(messages=[HumanMessage(content='How can I use LLM models to help in time series predictions?', additional_kwargs={}, response_metadata={}), AIMessage(content="Utilizing Large Language Models (LLMs) for time series predictions involves adapting their capabilities to effectively process and analyze temporal data. The first step is to choose an appropriate adaptation paradigm, which can be broadly categorized into embedding-visible and text-visible LLM adaptations. The embedding-visible approach involves redesigning the LLM to directly perceive time series embeddings, while the text-visible approach reformulates time series tasks into a textual format that the LLM can understand. This transformation allows numerical time series data to be represented as strings, enabling seamless integration into prompts that activate the LLM's predictive capabilities.\n\nOnce the adaptation paradigm is selected, the next step is to prepare the time series data for 

## Test FastAPI

- pip install fastapi uvicorn python-dotenv langchain-openai
- uvicorn fastapi_app:app --reload


In [6]:
import requests

response = requests.post("http://127.0.0.1:8000/ask", json={"question": "How can I use LLM models to help in time series predictions?"})
print(response.json()['response'])

To effectively utilize Large Language Models (LLMs) for time series predictions, it is essential to adapt their capabilities to the unique characteristics of temporal data. The first step involves selecting an appropriate adaptation paradigm, which can be broadly categorized into embedding-visible and text-visible approaches. The embedding-visible paradigm allows LLMs to directly process time series embeddings, leveraging their pre-trained knowledge while maintaining the integrity of the temporal data. In contrast, the text-visible paradigm reformulates time series tasks into a textual format, enabling LLMs to interpret numerical data as natural language inputs. This transformation facilitates the integration of time series data into the LLM's framework, allowing for more intuitive interactions and predictions.

Once you have chosen the adaptation strategy, the next step is to prepare your time series data for input into the LLM. This preparation may involve converting numerical time s

In [7]:
# Send message to chat
user_message = "Could you explain better the first step?"

response = requests.post("http://127.0.0.1:8000/ask", json={"question": user_message})
print(response.json()['response'])

The first step in utilizing Large Language Models (LLMs) for time series predictions involves selecting an appropriate adaptation paradigm that aligns with the nature of your temporal data. This selection is crucial as it determines how the LLM will process and interpret the time series information. There are two primary paradigms to consider: embedding-visible and text-visible approaches.

In the embedding-visible approach, the LLM is adapted to directly process time series embeddings. This means that the temporal data is transformed into a numerical representation that captures the underlying patterns and relationships within the time series. By leveraging the model's pre-trained knowledge, the LLM can utilize these embeddings to make predictions while preserving the integrity of the temporal data. This approach is particularly beneficial when dealing with large datasets, as it allows the model to learn from the rich features embedded in the time series without the need for extensive